In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, Dataset


train_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ColorJitter(brightness=0.2), 
    transforms.RandomRotation(10),          
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

eval_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])



raw_dataset = datasets.GTSRB(root='./data', split='train', download=True)


total_size = len(raw_dataset)
train_size = int(0.70 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size


generator = torch.Generator().manual_seed(42)
train_subset, val_subset, test_subset = random_split(
    raw_dataset, [train_size, val_size, test_size], generator=generator
)


class SubsetWrapper(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
        
    def __getitem__(self, index):
        image, label = self.subset[index]
        if self.transform:
            image = self.transform(image)
        return image, label
        
    def __len__(self):
        return len(self.subset)


train_data = SubsetWrapper(train_subset, transform=train_transforms)
val_data = SubsetWrapper(val_subset, transform=eval_transforms)
test_data = SubsetWrapper(test_subset, transform=eval_transforms)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

print(f"Data pipeline ready! Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

In [ ]:
import torch.nn as nn

class ModelA(nn.Module):
    def __init__(self, num_classes=43): 
        super().__init__()
        
        
        self.features = nn.Sequential(
            
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model_a = ModelA()


total_params = sum(p.numel() for p in model_a.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {total_params}")

## Data pipeline checks

Run these cells after the setup cell. They check the split sizes, image tensor shape and value range, and show examples from a training batch. The training images are augmented randomly, so the displayed examples can look a little different each time.

In [ ]:
from collections import Counter

expected_total = len(raw_dataset)
actual_total = len(train_data) + len(val_data) + len(test_data)
print(f"Raw images: {expected_total:,}")
print(f"Train / validation / test: {len(train_data):,} / {len(val_data):,} / {len(test_data):,}")
print(f"Split totals match: {actual_total == expected_total}")
print(f"Split proportions: train={len(train_data)/actual_total:.1%}, validation={len(val_data)/actual_total:.1%}, test={len(test_data)/actual_total:.1%}")
assert actual_total == expected_total
assert len(set(train_subset.indices) & set(val_subset.indices)) == 0
assert len(set(train_subset.indices) & set(test_subset.indices)) == 0
assert len(set(val_subset.indices) & set(test_subset.indices)) == 0
print("Splits are disjoint.")

In [ ]:
images, labels = next(iter(train_loader))
print(f"Image batch shape: {tuple(images.shape)}")
print(f"Label batch shape: {tuple(labels.shape)}")
print(f"Image values: min={images.min().item():.3f}, max={images.max().item():.3f}")
print(f"Labels in this batch: {sorted(labels.unique().tolist())}")
assert images.ndim == 4 and images.shape[1:] == (3, 64, 64)
assert labels.ndim == 1 and images.shape[0] == labels.shape[0]
assert torch.isfinite(images).all()
assert labels.min() >= 0 and labels.max() < 43
print("Batch checks passed.")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, image, label in zip(axes.flat, images[:8], labels[:8]):
    display_image = image.permute(1, 2, 0).cpu() * 0.5 + 0.5
    ax.imshow(display_image.clamp(0, 1))
    ax.set_title(f"Class {label.item()}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Model checks

The next cell sends one training batch through the network without training it. A successful result has one score for each of the 43 classes per image. The parameter count helps confirm the model fits your size budget.

In [ ]:
model_a.eval()
with torch.inference_mode():
    logits = model_a(images)

trainable_params = sum(parameter.numel() for parameter in model_a.parameters() if parameter.requires_grad)
print(f"Model output shape: {tuple(logits.shape)}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"All output values finite: {torch.isfinite(logits).all().item()}")
assert logits.shape == (images.shape[0], 43)
assert torch.isfinite(logits).all()
print("Model forward-pass checks passed.")

## Class coverage

This summary shows how many examples of each class are in each split. It helps reveal classes that are missing or unevenly represented. Because the split is random, inspect these counts before relying on validation results.

In [ ]:
def class_counts(dataset):
    counts = Counter()
    for _, label in dataset: 
        counts[int(label)] += 1
    return counts

train_counts = class_counts(train_subset)
val_counts = class_counts(val_subset)
test_counts = class_counts(test_subset)
print(f"Classes represented (train/validation/test): {len(train_counts)} / {len(val_counts)} / {len(test_counts)}")
print("Class | Train | Validation | Test")
for class_id in range(43):
    print(f"{class_id:>5} | {train_counts[class_id]:>5} | {val_counts[class_id]:>10} | {test_counts[class_id]:>4}")